## OCI Data Catalog API Demo

#### Prerequisites
1. create dynamic group matching rule - All {request.principal.type = 'aidataplatform',resource.compartment.id = '<compartment-ocid>'}
2. create policy - allow dynamic-group <dynamic-group> to manage data-catalog-family in compartment id <compartment-ocid>
3. setup requirements.txt with oci library configured (install in the compute server)
4. setup oci config and pem file in the compute server

In [76]:
import oci

COMPARTMENT_ID="ocid1.compartment.oc1..aaaaaaaay4b33s4cl7a3l3ehzklo3hsozhxcuuhil4nwnn5m76gjh4xypoua" 
config = oci.config.from_file("/Workspace/.oci/config")
oci.config.validate_config(config)
# Initialize service client with default config file
data_catalog_client = oci.data_catalog.DataCatalogClient(config)

In [77]:
# ------------------------------------------------------------------------
# List data catalogs
# ------------------------------------------------------------------------
list_catalogs_response = data_catalog_client.list_catalogs(
    compartment_id=COMPARTMENT_ID)

# Get the data from response
ids = [catalog.id for catalog in list_catalogs_response.data]
#print("Data catalog instance OCID: "+str(ids))

#print(list_catalogs_response.data)
for catalog in list_catalogs_response.data:
    print(f"Catalog key={catalog.id}\tCatalog display_name={catalog.display_name}")

# ------------------------------------------------------
# Display catalog attributes based on data catalog OCID
# ------------------------------------------------------
get_catalog_response = data_catalog_client.get_catalog(
    catalog_id=ids)

# Get the data from response
display_name = get_catalog_response.data.display_name
ocid=get_catalog_response.data.id
lifecycle_state=get_catalog_response.data.lifecycle_state
print("Name: "+display_name+" OCID: "+ocid+" lifecycle_state: "+lifecycle_state)

Catalog key=ocid1.datacatalog.oc1.us-chicago-1.amaaaaaa7ratczialrxpvt7va4hidvuh54lz2yboz44acxoc3btotzrwp63q	Catalog display_name=Pilot
Name: Pilot OCID: ocid1.datacatalog.oc1.us-chicago-1.amaaaaaa7ratczialrxpvt7va4hidvuh54lz2yboz44acxoc3btotzrwp63q lifecycle_state: ACTIVE


In [80]:
# ---------------------------------------------------------------
# List data catalog assets
# ----------------------------------------------------------------
list_data_assets_response = data_catalog_client.list_data_assets(
    catalog_id=ids)

# Get the data from response
#print(list_data_assets_response.data)
for asset in list_data_assets_response.data.items:
    print(f"Asset key={asset.key}\tAsset display_name={asset.display_name}") if "psft" in asset.display_name.lower() else None

Asset key=a607d5a3-d287-4a1c-b2f1-dc5a27fb1f43	Asset display_name=PSFT
Asset key=7c8b8f91-498f-49cb-9c99-aa2bca998042	Asset display_name=PSFTCLN


In [81]:
# ----------------------------------------------
# List data catalog entities for PSFT data asset
# ----------------------------------------------
ASSET_KEY="a607d5a3-d287-4a1c-b2f1-dc5a27fb1f43"
list_entities_response = data_catalog_client.list_entities(
    catalog_id=ids,
    data_asset_key=ASSET_KEY)
list_entities_response = sorted(
    list_entities_response.data.items,
    key=lambda e: (e.display_name or "").lower()
)

# Get the data from response
#print(list_entities_response.data)
for entity in list_entities_response[:10]: # Print ordered upto n rows
    print(f"Key={entity.key}\tEntity={entity.display_name}\tSource={entity.external_key}")

Key=1c00adae-a888-4c0f-a21c-bbdb84dce91d	Entity=PS_ACCOUNT_TBL	Source=mimicIII/PSFT/PS_ACCOUNT_TBL
Key=e434c244-f4a9-484c-b8e4-406b20b16d02	Entity=PS_ADDRESSES	Source=mimicIII/PSFT/PS_ADDRESSES
Key=486c0531-2509-4fa0-96d8-ab60ede5147b	Entity=PS_COMPENSATION	Source=mimicIII/PSFT/PS_COMPENSATION
Key=dc015272-3292-45c8-82d9-c63ef1262f4f	Entity=PS_COMPENSATION_HST	Source=mimicIII/PSFT/PS_COMPENSATION_HST
Key=9bc8521e-aa60-450f-ba49-bb800f190b3e	Entity=PS_CUST_TBL	Source=mimicIII/PSFT/PS_CUST_TBL
Key=87654206-2627-449f-81cf-dcde80be83e9	Entity=PS_DEPT_TBL	Source=mimicIII/PSFT/PS_DEPT_TBL
Key=12294bf8-8352-405b-a71f-9feea552a1b3	Entity=PS_EMAIL_ADDRESSES	Source=mimicIII/PSFT/PS_EMAIL_ADDRESSES
Key=7e297acd-9420-4201-a9d6-0d38e648b6df	Entity=PS_EMPLOYMENT	Source=mimicIII/PSFT/PS_EMPLOYMENT
Key=408e3950-0367-47b6-888c-c25176ce9a81	Entity=PS_FUND_TBL	Source=mimicIII/PSFT/PS_FUND_TBL
Key=99521dee-6885-4b43-b23c-be0649af5b2a	Entity=PS_INV_ITEM_MASTER	Source=mimicIII/PSFT/PS_INV_ITEM_MASTER


In [82]:
# ----------------------------------------------------------------------
# List all Glossaries defined in the data catalog
# ----------------------------------------------------------------------
# Initialize service client with default config file
list_glossaries_response = data_catalog_client.list_glossaries(
    catalog_id=ids)

# Get the data from response
#print(list_glossaries_response.data)
for glossary in list_glossaries_response.data.items:
    print(f"Key={glossary.key}\tGlossary={glossary.display_name}")

Key=06c12461-d78c-4d0a-978f-6124f09abb44	Glossary=PSFT_Business_Glossary
Key=705e4631-6787-4073-9efe-9b2b1d5a4bb4	Glossary=Ellison-Business-Glossary


In [83]:
# ----------------------------------------------------------------------
# List all namespaces for PSFT custom properties
# ----------------------------------------------------------------------
list_namespaces_response = data_catalog_client.list_namespaces(
    catalog_id=ids,)

# Get the data from response
#print(list_namespaces_response.data)
for namespace in list_namespaces_response.data.items:
    print(f"Key={namespace.key}\tNamespace={namespace.display_name}")

Key=0e4d60d9-d5b5-467f-89bb-22db63a3ee18	Namespace=Custom Properties


#### Updating custom properties for an entity

In [93]:
# ----------------------------------------------------------------------
# List all custom properties for PSFT for the namespace
# ----------------------------------------------------------------------
NAMESPACE_ID="0e4d60d9-d5b5-467f-89bb-22db63a3ee18" # from above
list_custom_properties_response = data_catalog_client.list_custom_properties(
    catalog_id=ids,
    namespace_id=NAMESPACE_ID,)

# Get the data from response
#print(list_custom_properties_response.data)
for cp in list_custom_properties_response.data.items:
    print(f"Namespace={NAMESPACE_ID}\tKey={cp.key}\tCustom property={cp.display_name}") if "PSFT" in cp.display_name.upper() else None

Namespace=0e4d60d9-d5b5-467f-89bb-22db63a3ee18	Key=94f9cb3e-173b-43cb-95c3-479bf9961df3	Custom property=PSFT
Namespace=0e4d60d9-d5b5-467f-89bb-22db63a3ee18	Key=0511d6cf-afb5-4499-a83d-fcaab68f54f5	Custom property=PSFT_MOD


In [92]:
# ----------------------------------------------------------------------
# List allowed values for a custom property
# ----------------------------------------------------------------------
CP_KEY="94f9cb3e-173b-43cb-95c3-479bf9961df3" # from above for PSFT

get_custom_property_response = data_catalog_client.get_custom_property(
    catalog_id=ids,
    namespace_id=NAMESPACE_ID,
    custom_property_key="94f9cb3e-173b-43cb-95c3-479bf9961df3",
    )

# Get the data from response
print(f"Name={get_custom_property_response.data.display_name}\tAllowed values={get_custom_property_response.data.allowed_values}")

Name=PSFT	Allowed values=["HCM", "FIN", "SCM", "COM"]


In [96]:
# ----------------------------------------------------------------------
# Getting custom properties LOV and assigned values for an Entity
# ----------------------------------------------------------------------
ENTITY_KEY="1c00adae-a888-4c0f-a21c-bbdb84dce91d"
get_entity_response = data_catalog_client.get_entity(
    catalog_id=ids,
    data_asset_key=ASSET_KEY,
    entity_key=ENTITY_KEY,
   )
#print(get_entity_response.data.custom_property_members)
for cp in get_entity_response.data.custom_property_members:
    print(f"key={cp.key}\tname={cp.display_name}\tallowed_values={cp.allowed_values}\tnamespace={cp.namespace_name}\tAssigned={cp.value}")


key=a80585a1-1d40-4a23-8e98-6b77d79407bf	name=PSFT	allowed_values=["HCM", "FIN", "SCM", "COM"]	namespace=Custom Properties	Assigned=HCM
key=ba1dea89-3acd-4da9-84f5-448a6f94cbfa	name=PSFT_MOD	allowed_values=["PSFT_MAIN", "PSFT_CLINICAL"]	namespace=Custom Properties	Assigned=PSFT_MAIN


In [95]:
# ----------------------------------------------------------------------
# Updating custom properties assigned values for an Entity
# ----------------------------------------------------------------------
ENTITY_KEY="1c00adae-a888-4c0f-a21c-bbdb84dce91d"
NEW_PSFT="HCM"
NEW_PSFT_MOD="PSFT_MAIN"
update_entity_response = data_catalog_client.update_entity(
    catalog_id=ids,
    data_asset_key=ASSET_KEY,
    entity_key=ENTITY_KEY,
    update_entity_details=oci.data_catalog.models.UpdateEntityDetails(
        custom_property_members=[
            oci.data_catalog.models.CustomPropertySetUsage(
                    key="a80585a1-1d40-4a23-8e98-6b77d79407bf",
                    display_name="PSFT",
                    value=NEW_PSFT,
                    namespace_name="Custom Properties"
                    ),
            oci.data_catalog.models.CustomPropertySetUsage(
                    display_name="PSFT_MOD",
                    value=NEW_PSFT_MOD,
                    namespace_name="Custom Properties"
                    )],)
        )

In [62]:
# Get the data from response
#print(update_entity_response.data)

#### Checking out harvest jobs and incremental executions

In [100]:
# ----------------------------------------------------------------------
# List all job definitions for the PSFTCLN data asset
# ----------------------------------------------------------------------
ASSET_KEY="7c8b8f91-498f-49cb-9c99-aa2bca998042"
DISPLAY_CONTAINS="Harvest"
list_job_definitions_response = data_catalog_client.list_job_definitions(
    catalog_id=ids,
    data_asset_key=ASSET_KEY,
    display_name_contains=DISPLAY_CONTAINS)

# Get the data from response
#print(list_job_definitions_response.data)
for jobdef in list_job_definitions_response.data.items:
    print(f"Key={jobdef.key}\tName={jobdef.display_name}\tJob type={jobdef.job_type}\tStatus={jobdef.job_execution_state}\tlast execution={jobdef.time_latest_execution_ended}")

Key=52682f89-aef2-4540-bf31-2b590f0011ae	Name=Harvest_PSFTCLN_20260430123025	Job type=HARVEST	Status=SUCCEEDED	last execution=2026-05-06 20:58:41.161993+00:00


In [102]:
# ----------------------------------------------------------------------
# List jobs
# ----------------------------------------------------------------------
list_jobs_response = data_catalog_client.list_jobs(
    catalog_id=ids,
    data_asset_key=ASSET_KEY,
    display_name_contains=DISPLAY_CONTAINS)

# Get the data from response
#print(list_jobs_response.data)
for job in list_jobs_response.data.items:
    print(f"Job Key={job.key}\tName={job.display_name}\tStatus={lifecycle_state}\tExec Count={job.execution_count}\tStatus={job.time_of_latest_execution}")

Job Key=387128b1-59d6-4414-9654-8d2fd2da630a	Name=Harvest_PSFTCLN_20260430123025_job	Status=ACTIVE	Exec Count=7	Status=2026-05-06 20:58:37.487847+00:00


In [103]:
JOB_KEY="387128b1-59d6-4414-9654-8d2fd2da630a"
JOB_TYPE="HARVEST"
list_job_executions_response = data_catalog_client.list_job_executions(
    catalog_id=ids,
    job_key=JOB_KEY,
    job_type=JOB_TYPE)

# Get the data from response
#print(list_job_executions_response.data)
for jobexec in list_job_executions_response.data.items:
    print(f"Job Key={jobexec.job_key}\tJob Exec key={jobexec.key}\tType={jobexec.job_type}\tStatus={lifecycle_state}\time ended={jobexec.time_ended}")

Job Key=387128b1-59d6-4414-9654-8d2fd2da630a	Job Exec key=39d2cc15-6529-49bd-a5b0-7cb5743af35e	Type=HARVEST	Status=ACTIVE	ime ended=2026-05-06 20:48:27.655963+00:00
Job Key=387128b1-59d6-4414-9654-8d2fd2da630a	Job Exec key=41f882b8-d5fd-4f33-87c2-9f7f765c9f7f	Type=HARVEST	Status=ACTIVE	ime ended=2026-05-06 20:46:14.843199+00:00
Job Key=387128b1-59d6-4414-9654-8d2fd2da630a	Job Exec key=772add67-f9cd-4435-aa2c-26f73532e5d3	Type=HARVEST	Status=ACTIVE	ime ended=2026-05-06 20:58:41.161993+00:00
Job Key=387128b1-59d6-4414-9654-8d2fd2da630a	Job Exec key=c6444225-cd9a-4d00-9ed4-f8c67aa4b8eb	Type=HARVEST	Status=ACTIVE	ime ended=2026-05-06 20:50:14.495887+00:00
Job Key=387128b1-59d6-4414-9654-8d2fd2da630a	Job Exec key=cd4142c4-5e09-4256-8318-0bba00b9eee2	Type=HARVEST	Status=ACTIVE	ime ended=2026-05-06 20:30:21.324807+00:00
Job Key=387128b1-59d6-4414-9654-8d2fd2da630a	Job Exec key=e3c3bcff-c05b-4bef-b2bc-33599a016395	Type=HARVEST	Status=ACTIVE	ime ended=2026-05-06 20:54:33.835230+00:00
Job Key=38

In [66]:
# ----------------------------------------------------------------------
# Execute an existing harvest job 
# ----------------------------------------------------------------------
create_job_execution_response = data_catalog_client.create_job_execution(
    catalog_id=ids,
    job_key=JOB_KEY,
    create_job_execution_details=oci.data_catalog.models.CreateJobExecutionDetails(
        job_type=JOB_TYPE,
        ),
        )

# Get the data from response
job=create_job_execution_response.data
print(f"Job Key={job.job_key}\tJob Exec key={job.key}\tType={job.job_type}\tStatus={job.lifecycle_state}")

Job Key=d50e725f-a34d-414f-bcad-bf692ce4d62a	Job Exec key=4a2416fb-0f06-42a5-a682-66f0c8d42419	Type=HARVEST	Status=CREATED


In [119]:
# Initialize service client with default config file
data_catalog_client = oci.data_catalog.DataCatalogClient(config)


# Send the request to service, some parameters are not required, see API
# doc for more info
search_criteria_response = data_catalog_client.search_criteria(
    catalog_id=ids,  
    display_name="account",
    )
print(search_criteria_response.data)

{
  "count": 10000,
  "faceted_search_aggregation": null,
  "items": [
    {
      "attribute_type": null,
      "business_name": null,
      "created_by_id": "ocid1.user.oc1..aaaaaaaajz34bwifxsu6u455kya5gy2pzzaimtjy2wdmqzahcbk7prr67e3q",
      "custom_properties": null,
      "data_asset_key": "672dd4d1-04d8-4839-9016-23654bb6c41e",
      "data_asset_name": null,
      "data_asset_type": "Oracle Object Storage",
      "description": null,
      "entity_name": null,
      "entity_type": "File",
      "entitykey": null,
      "expression": null,
      "external_data_type": null,
      "external_type_name": null,
      "folder_key": "a49613ef-9181-419c-ba47-f864d7c9553a",
      "folder_name": null,
      "folder_type": "Bucket",
      "glossary_key": null,
      "glossary_name": null,
      "key": "a7972b91-9c66-474e-a692-ae0df1030fc9",
      "lifecycle_state": null,
      "name": "data/design-info.json.30",
      "parent_term_key": null,
      "parent_term_name": null,
      "path": "El